In [13]:
import math 
from collections import Counter

In [14]:
# fungsi dan prosedur

def entropy(data):
    n = len(data)
    nGrup = Counter(data)

    ent = 0.0

    for c in nGrup:
        p = nGrup[c] / n
        ent -= p * math.log2(p)

    return ent

# gain 

def gain(data_input,data_output,fitur):
    base_ent = entropy(data_output)
    base_n = len(data_output)

    # kelompok berdasarkan nilai atribut
    subsets = {}
    for x,label in zip(data_input, data_output):
        key = x[fitur]
        subsets.setdefault(key, []).append(label)
    subset_ent = 0.0
    for label in subsets.values():
        subset_ent += (len(label) / base_n) * entropy(label)

    return base_ent - subset_ent

# prediksi label untuk satu data
def semua_daun(node):
    if not isinstance(node, dict):
        return [node]
    hasil = []
    for v in node.values():
        hasil.extend(semua_daun(v))
    return hasil

def prediksi(tree, data):
    if not isinstance(tree, dict):
        return tree
    idx = next(iter(tree))
    cabang = tree[idx]
    if data[idx] in cabang:
        return prediksi(cabang[data[idx]], data)
    # nilai fitur tidak dikenal -> pakai daun terbanyak di cabang itu
    return Counter(semua_daun(cabang)).most_common(1)[0][0]

# akurasi
def akurasi(y_asli, y_pred):
    benar = sum(1 for a, b in zip(y_asli, y_pred) if a == b)
    return benar / len(y_asli)

# RMSE
def rmse(y_asli, y_pred):
    se = sum((a - b) ** 2 for a, b in zip(y_asli, y_pred))
    return math.sqrt(se / len(y_asli))



In [15]:
# Tree
def buat_tree(data_input, data_output, feature_indices):
    # semua output (LabeL) sama -> Leaf
    if len(set(data_output)) == 1:
        return data_output[0]

    # fitur kosong -> habis
    if not feature_indices:
        return Counter(data_output).most_common(1)[0][0]

    # Pilih fitur terbaik berdasarkan gain
    gains = [(gain(data_input, data_output, idx), idx) for idx in feature_indices]
    gains.sort(reverse=True)
    best_gain, best_fitur = gains[0]

    # Jika gain = 0 -> tidak ada peningkatan
    if best_gain == 0:
        return Counter(data_output).most_common(1)[0][0]

    tree = {best_fitur: {}}

    # Nilai-nilai unik fitur pada data saat ini
    values = set(x[best_fitur] for x in data_input)
    for val in values:
        # buat subset data untuk nilai fitur val
        sub_input = [x for x, label in zip(data_input, data_output) if x[best_fitur] == val]
        sub_output = [label for x, label in zip(data_input, data_output) if x[best_fitur] == val]

        # jika subset kosong
        if not sub_input:
            tree[best_fitur][val] = Counter(data_output).most_common(1)[0][0]
        else:
            sisa_fitur = [i for i in feature_indices if i != best_fitur]
            tree[best_fitur][val] = buat_tree(sub_input, sub_output, sisa_fitur)

    return tree

In [16]:
# inisialisasi data

import pandas as pd
import kagglehub
from pathlib import Path

# download dataset dari kaggle (sekali download, run berikutnya pakai cache)
path = kagglehub.dataset_download("ddosad/ecommerce-customer-service-satisfaction")

# cari file csv di folder hasil download
data = next(Path(path).glob("*.csv"), None)
if data is None:
    raise FileNotFoundError("file csv tidak ditemukan di hasil download kaggle")

# membaca data (sampling 10000 baris supaya training cepat)
df = pd.read_csv(data, nrows=10000)

# praprocessing data
# encoding setiap fitur kategorikal
fitur = ["channel_name", "category", "Tenure Bucket", "Agent Shift"]
target = "CSAT Score"

# buang baris yang fitur/targetnya kosong
df = df[fitur + [target]].dropna()

# kamus encoding dibangun dari nilai unik tiap kolom (urut, biar deterministik)
kamus = {}
for kolom in fitur:
    nilai = sorted(df[kolom].astype(str).unique())
    kamus[kolom] = {v: i + 1 for i, v in enumerate(nilai)}

# mengubah data kategorikal menjadi angka
for kolom, mapping in kamus.items():
    df[kolom] = df[kolom].astype(str).map(mapping)

# target: puas (1) kalau CSAT >= 4, selain itu tidak puas (0)
df["Puas"] = (df[target] >= 4).astype(int)

# mengambil fitur sebagai list
x = df[fitur].values.tolist()

# mengambil target sebagai list
y = df["Puas"].values.tolist()

# mengubah nilai menjadi integer
x = [[int(i) for i in row] for row in x]
y = [int(i) for i in y]

print(len(x), len(y))
print(x[:3], y[:3])

10000 10000
[[3, 9, 5, 3], [3, 9, 4, 3], [2, 6, 5, 2]] [1, 1, 1]


In [17]:
# main program

feature_indices = list(range(len(x[0])))
tree = buat_tree(
    x, 
    y, 
    feature_indices
)

print("decision tree:", tree)



decision tree: {1: {1: 1, 2: {3: {1: {2: {1: 1, 2: 1, 3: {0: {2: 0, 3: 1}}, 4: {0: {2: 1, 3: 1}}}}, 2: {2: {1: {0: {2: 1, 3: 1}}, 2: {0: {1: 1, 2: 1, 3: 0}}, 3: {0: {2: 1, 3: 1}}, 4: {0: {2: 1, 3: 1}}, 5: {0: {2: 1, 3: 1}}}}, 3: {2: {1: {0: {1: 1, 2: 1, 3: 1}}, 2: {0: {1: 1, 2: 1, 3: 1}}, 3: {0: {2: 0, 3: 1}}, 4: {0: {1: 0, 2: 1, 3: 1}}, 5: {0: {1: 1, 2: 0, 3: 1}}}}, 4: 1, 5: {2: {2: 1, 3: 1, 4: {0: {1: 1, 2: 1, 3: 1}}}}}}, 3: {2: {1: {0: {1: 1, 2: {3: {1: 0, 2: 1, 3: 1}}, 3: 1}}, 2: {3: {1: 1, 2: {0: {2: 1, 3: 0}}, 3: {0: {1: 1, 2: 1, 3: 0}}, 4: 1}}, 3: {0: {1: 1, 2: {3: {1: 1, 2: 1, 3: 1, 4: 1}}, 3: 1}}, 4: {0: {1: {3: {1: 0, 3: 1, 5: 1}}, 2: {3: {1: 1, 2: 1, 3: 1, 4: 0, 5: 1}}, 3: {3: {1: 0, 2: 0, 3: 1, 5: 1}}}}, 5: {3: {2: {0: {2: 1, 3: 1}}, 3: 1}}}}, 4: {3: {1: 1, 2: {2: {1: 1, 2: {0: {2: 1, 3: 1}}, 3: 1, 4: 1, 5: 1}}, 3: {2: {1: 1, 2: {0: {1: 1, 2: 1}}, 3: 1, 4: {0: {1: 1, 2: 1, 3: 1}}, 5: 1}}, 4: {2: {3: 1, 4: 0}}, 5: {0: {2: 1, 3: 1}}}}, 5: 1, 6: {2: {1: {3: {1: 0, 2: {0: {1: 0

In [18]:
# langkah pengujian: split data latih/uji 80/20, hitung akurasi dan RMSE
import random
random.seed(42)
idx = list(range(len(x)))
random.shuffle(idx)
batas = int(0.8 * len(x))
train_idx, test_idx = idx[:batas], idx[batas:]

x_latih = [x[i] for i in train_idx]
y_latih = [y[i] for i in train_idx]
x_uji = [x[i] for i in test_idx]
y_uji = [y[i] for i in test_idx]

# bangun pohon dari data latih
tree_besar = buat_tree(x_latih, y_latih, list(range(len(x_latih[0]))))

# prediksi data uji
y_pred = [prediksi(tree_besar, xi) for xi in x_uji]

# hitung akurasi dan RMSE pada data uji
acc = akurasi(y_uji, y_pred)
err = rmse(y_uji, y_pred)

print("data latih:", len(x_latih), "| data uji:", len(x_uji))
print("akurasi data uji: %.4f (%.2f%%)" % (acc, acc * 100))
print("RMSE data uji: %.4f" % err)

data latih: 8000 | data uji: 2000
akurasi data uji: 0.8205 (82.05%)
RMSE data uji: 0.4237
